# Titanic Data Analysis

## 1. Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Loading the Dataset

In [ ]:
df_original = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
df = df_original.copy()

df.head()

## 3. Initial Data Exploration

The dataset structure and missing values were examined.

In [ ]:
df.info()

In [ ]:
df.isna().sum()

- Age contains missing values
- Cabin has many missing values
- Embarked contains a few missing values

## 4. Data Cleaning

The `Cabin` variable was removed because it contained a large proportion of missing values.

In [ ]:
df = df.drop('Cabin', axis=1)

Missing values in the `Age` variable were filled using the median because it handles extreme values better.

In [ ]:
df['Age'] = df['Age'].fillna(df['Age'].median())

Missing values in the `Embarked` variable were imputed using the mode because it is a categorical variable.

In [ ]:
df['Embarked'] = df["Embarked"].fillna(df["Embarked"].mode()[0])

Outliers in the `Age` variable were retained because they represent valid passenger ages.


In [ ]:
Q1 = df['Age'].quantile(0.25)
Q3 = df['Age'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df['Age']<lower) | (df['Age']>upper)]
outliers

Missing value check after cleaning:

In [ ]:
df.isnull().sum()

## 5. Feature Engineering
New variables were created to better capture survival-related patterns.

#### `Age_Group` Feature
Passengers were grouped by age ranges.

In [ ]:
max_age = int(df['Age'].max())

df['Age_Group'] = pd.cut(df['Age'], bins=[0,12,18,30,50,max_age], labels=['[0-12]', '(12-18]', '(18-30]', '(30-50]', f'(50-{max_age}]'], include_lowest=True)
df.groupby('Age_Group', observed=True)['Survived'].mean()

#### `FamilySize` Feature
A new feature was created to represent the total family size aboard the ship.

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

#### `IsAlone` Feature
Passengers traveling alone were identified.

In [ ]:
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

Check the new features:

In [ ]:
df[["Age", "Age_Group", "FamilySize", "IsAlone"]].head()

In [ ]:
df[["Age", "Age_Group", "FamilySize", "IsAlone"]].describe()

## 6. Exploratory Data Analysis (EDA)

### Survival Rate by Gender

In [ ]:
plt.figure(figsize=(6,4))

sns.barplot(data=df, x='Sex', y='Survived', hue='Sex', palette='Set1', errorbar=None)

plt.title('Survival Rate by Gender')
plt.ylabel('Survival Rate')

plt.grid(axis='y', linestyle='--', alpha=0.2)

plt.tight_layout()
plt.show()

(df.groupby('Sex')['Survived'].mean() * 100).round(2)

Female passengers had a significantly higher survival rate than males. Approximately 74% of female passengers survived, compared to only about 19% of male passengers, indicating that gender was strongly related to survival outcomes.

### Survival by Passenger Class

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(data=df, x='Pclass', hue='Survived', palette='Set2')
plt.title('Survival by Passenger Class')
plt.grid(axis='y', linestyle='--', alpha=0.2)
plt.tight_layout()

Passengers in higher classes were more likely to survive compared to those in lower classes.

### Survival by Age Group

In [ ]:
plt.figure(figsize=(7,4))

sns.countplot(
    data=df,
    x='Age_Group',
    hue='Survived',
    palette='Set2'
)

plt.title('Survival by Age Group')
plt.grid(axis='y', linestyle='--', alpha=0.2)
plt.tight_layout()

Survival rates vary across different age groups. The 18–30 age group is the largest group in the dataset and also has the highest number of deaths. In all age groups, the number of passengers who did not survive is higher than the number of survivors.

### Correlation Heatmap

In [ ]:
plt.figure(figsize=(10,6))

sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm',fmt='.2f')

plt.title('Correlation Heatmap')
plt.tight_layout()

The heatmap indicates that `Pclass` has a moderate negative correlation with `Survived`, suggesting that passengers in higher classes were more likely to survive. `Fare` shows a weak positive relationship with `Survived`, while `IsAlone` has a weak negative correlation with survival outcomes.

### Fare by Survival

In [ ]:
plt.figure(figsize=(7,4))

sns.violinplot(data=df, x='Survived', y='Fare', hue='Sex', split=True, palette='Set1')

plt.title('Fare by Survival')
plt.tight_layout()

Passengers with higher fares were generally more likely to survive. Female passengers also appear to have better survival outcomes across different fare ranges.

### Survival Rate by IsAlone

In [ ]:
plt.figure(figsize=(6,4))

sns.barplot(data=df, x='IsAlone', y='Survived', hue='IsAlone', palette='Set2', legend=False)

plt.title('Survival Rate by IsAlone')
plt.ylabel('Survival Rate')
plt.grid(axis='y', linestyle='--', alpha=0.2)
plt.tight_layout()

Passengers traveling alone appear to have slightly higher survival rates compared to passengers traveling with family members.

### Family Size and Survival Rate

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,4))

sns.countplot(data=df, x='FamilySize', hue='FamilySize', palette='flare', legend=False, ax=ax[0])
ax[0].grid(axis='y', linestyle='--', alpha=0.2)
ax[0].set_title('Family Size Distribution')

sns.barplot(data=df, x='FamilySize', y='Survived', hue='FamilySize', palette='flare', legend=False, errorbar=None, ax=ax[1])
ax[1].grid(axis='y', linestyle='--', alpha=0.2)
ax[1].set_title('Survival Rate by Family Size')
ax[1].set_ylabel('Survival Rate')

sns.despine()
plt.tight_layout()

Most passengers traveled alone or with small families. Survival rates vary across different family sizes, with medium-sized families showing relatively higher survival rates compared to some larger family groups.